# Model Testing and Feature Manipulation

In [52]:
# Import Python Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import leafmap.foliumap as leafmap
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [53]:
df = pd.read_excel('airline_ticket_dataset.xlsx')
df_clean = df.dropna()

### Phase 1: The Core Physics of a Ticket (Distance and Volume)
Every flight starts with a basic cost structure. Before we look at competition, we must account for how far the plane is going and how many people are on the route.

In [54]:
# 1. Create the Log Variables
df_clean['log_fare'] = np.log(df_clean['fare'])
df_clean['log_dist'] = np.log(df_clean['nsmiles'])
df_clean['log_pax'] = np.log(df_clean['passengers'])

# 2. Prepare the Features
# We keep market share and LCC percentage as they are (since they are already 0-1)
X = df_clean[['log_dist', 'log_pax']]
X = sm.add_constant(X)
y = df_clean['log_fare']

# 3. Fit the Log-Log Model
log_model = sm.OLS(y, X).fit()
print(log_model.summary())

                            OLS Regression Results                            
Dep. Variable:               log_fare   R-squared:                       0.370
Model:                            OLS   Adj. R-squared:                  0.370
Method:                 Least Squares   F-statistic:                     4103.
Date:                Sat, 28 Feb 2026   Prob (F-statistic):               0.00
Time:                        23:05:06   Log-Likelihood:                 1761.2
No. Observations:               13972   AIC:                            -3516.
Df Residuals:                   13969   BIC:                            -3494.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          3.8558      0.024    160.147      0.0

We began our analysis by establishing the "baseline" cost of air travel. In the airline industry, distance is the primary driver of cost due to fuel and labor. However, we used logarithmic transformations for both distance and fare. This is important because it acknowledges "economies of scale"—the idea that the cost of flying doesn't increase by a flat dollar amount per mile, but rather tapers off as flights get longer. By also including log_pax, we account for market density, as high-volume routes often enjoy lower per-passenger costs.

### Phase 2: Identifying the "Disruptive Force"
Now that we have the basics, we need to measure how much Low-Cost Carriers (LCCs) actually break the pricing power of major airlines.

In [55]:
# Variable: Your LCC Metric (Disruptive Force)
# Multiplies the low-fare share by total passengers to find the volume of cheap seats
df_clean['lcc_volume'] = df_clean['lf_ms'] * df_clean['passengers']

# Variable: Airline and Year effects
# We turn these into 1s and 0s so the model knows which carrier is flying the route and which year the ticket was purchased
carrier_dummies = pd.get_dummies(df_clean['carrier_lg'], prefix='Airline').astype(int)
year_dummies = pd.get_dummies(df_clean['Year'], prefix='Year').astype(int)
quarter_dummies = pd.get_dummies(df_clean['quarter'], prefix='q', drop_first=True).astype(int)

# Fit the OLS Regression
X = pd.concat([
    df_clean[['log_dist', 'log_pax', 'lcc_volume']]
], axis=1)
X = sm.add_constant(X) # Add the base intercept
y = df_clean['log_fare']

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:               log_fare   R-squared:                       0.374
Model:                            OLS   Adj. R-squared:                  0.374
Method:                 Least Squares   F-statistic:                     2780.
Date:                Sat, 28 Feb 2026   Prob (F-statistic):               0.00
Time:                        23:05:07   Log-Likelihood:                 1802.7
No. Observations:               13972   AIC:                            -3597.
Df Residuals:                   13968   BIC:                            -3567.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          3.7916      0.025    151.553      0.0

To measure competition, we moved beyond simple percentages and created a variable called LCC Volume. A standard 10% market share for a budget airline is just a number, but a 10% share on a route with 10,000 daily passengers represents a massive physical supply of cheap seats. By multiplying the low-fare carrier's market share by the total passenger count, we created an indicator that represents the "Disruptive Force" in a market. This makes sense for this case because it captures the actual pressure placed on legacy carriers to lower their prices to prevent losing thousands of customers to a cheaper neighbor.

### Phase 3: The Monopoly Power

Not all routes are equal. Some are protected by the market dominance of a hub, where an airline can maintain high prices despite competition.

In [56]:
# Create the Hub Power variable for both endpoints
df_clean['hub_power_1'] = df_clean['large_ms'] * df_clean['TotalPerPrem_city1']
df_clean['hub_power_2'] = df_clean['large_ms'] * df_clean['TotalPerPrem_city2']

# Regress fare on the hub power variables
X = df_clean[['hub_power_1', 'hub_power_2']]
X = sm.add_constant(X)
y = df_clean['log_fare']

model_hub = sm.OLS(y, X).fit()
print(model_hub.summary())

                            OLS Regression Results                            
Dep. Variable:               log_fare   R-squared:                       0.225
Model:                            OLS   Adj. R-squared:                  0.224
Method:                 Least Squares   F-statistic:                     2023.
Date:                Sat, 28 Feb 2026   Prob (F-statistic):               0.00
Time:                        23:05:07   Log-Likelihood:                 309.29
No. Observations:               13972   AIC:                            -612.6
Df Residuals:                   13969   BIC:                            -589.9
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const           5.4612      0.002   2626.207      

To balance our model, we had to account for Hub Power. In the U.S. aviation market, an airline's pricing power is significantly amplified if they control the "Hub" at either end of the trip. We calculated this as an interaction term between market share and hub status. This indicator is vital because it explains why some monopolies are "expensive monopolies"—they aren't just the biggest carrier; they own the gates and the slots, making it nearly impossible for a low-cost carrier to truly disrupt them.

### Phase 4: Finalizing the Model with Brand and Time Adjustments
We have to recognize that some airlines are just cheaper by design (Brand) and some years were more expensive for everyone (Economy).

In [57]:
# Variable: Airline and Year effects
# We turn these into 1s and 0s so the model knows which carrier is flying the route and which year the ticket was purchased
carrier_dummies = pd.get_dummies(df_clean['carrier_lg'], prefix='Airline').astype(int)
year_dummies = pd.get_dummies(df_clean['Year'], prefix='Year').astype(int)
quarter_dummies = pd.get_dummies(df_clean['quarter'], prefix='q', drop_first=True).astype(int)

# Fit the OLS Regression
X = pd.concat([
    carrier_dummies, 
    year_dummies, 
    quarter_dummies
], axis=1)
X = sm.add_constant(X) # Add the base intercept
y = df_clean['log_fare']

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:               log_fare   R-squared:                       0.238
Model:                            OLS   Adj. R-squared:                  0.237
Method:                 Least Squares   F-statistic:                     242.4
Date:                Sat, 28 Feb 2026   Prob (F-statistic):               0.00
Time:                        23:05:07   Log-Likelihood:                 433.36
No. Observations:               13972   AIC:                            -828.7
Df Residuals:                   13953   BIC:                            -685.4
Df Model:                          18                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       1.053e+11   1.76e+11      0.597      0.5

Our final narrative concludes by adjusting for Fixed Effects. By including Carrier and Year Dummies, we are "leveling the playing field." These variables act like on/off switches that allow the model to recognize that a ticket on Spirit is naturally cheaper than a ticket on United, regardless of distance. Similarly, the Year Dummies account for macro-economic shifts like fuel inflation. By stripping away these brand and timing biases, we are left with a clear, accurate view of how distance, volume, and competitive disruption actually dictate the price of a ticket in today's market.

### Phase 5: Putting it All Together
In this final stage, we bring every predictor of our feature engineering together to explain the complex reality of airline pricing.

In [58]:
# Regress fare on all new variables
X = pd.concat([
    df_clean[['lcc_volume', 'log_dist', 'log_pax', 'hub_power_1', 'hub_power_2']], 
    carrier_dummies, 
    year_dummies, 
    quarter_dummies
], axis=1)
X = sm.add_constant(X)
y = df_clean['log_fare']

model_hub = sm.OLS(y, X).fit()
print(model_hub.summary())

                            OLS Regression Results                            
Dep. Variable:               log_fare   R-squared:                       0.727
Model:                            OLS   Adj. R-squared:                  0.727
Method:                 Least Squares   F-statistic:                     1690.
Date:                Sat, 28 Feb 2026   Prob (F-statistic):               0.00
Time:                        23:05:07   Log-Likelihood:                 7607.5
No. Observations:               13972   AIC:                        -1.517e+04
Df Residuals:                   13949   BIC:                        -1.500e+04
Df Model:                          22                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const           2.8262      0.013    219.460      

This final model represents our full market analysis. By synthesizing every variable, we can explain the complex reality of airline pricing as the result of three primary factors:
- Operational Reality: The cost of the distance and the efficiency of a high-volume route (log_dist and log_pax).
- Competitive Friction: The downward pressure exerted by the physical presence of low-cost seats (lcc_volume).
- Structural Power: The premium an airline can extract through its "Fortress Hub" and market share (hub_power).

Our model shows that airfare is a direct result of the competition between an airline’s structural control and a passenger's available choices. On routes touching highly dominant hubs, we observe a clear price premium. This is because dominant carriers control the essential infrastructure—like gates and flight slots—creating a monopoly effect that allows them to charge more due to limited local competition. In these scenarios, a higher market share for the leading airline almost always translates to higher average fares.

However, we found that Low-Cost Carrier (LCC) Volume acts as the primary equalizer. It is not just the presence of a budget airline that lowers prices, but the actual quantity of seats they offer. As this volume of affordable seats increases, it creates a "price ceiling" that forces even the most dominant airlines to lower their fares to remain competitive.

Ultimately, the market fare is determined by which force is stronger: the Monopoly Power of the hub leader or the Disruptive Force of the low-cost competitors. Our results suggest that the most effective way to lower fares across the industry is to support the expansion of low-cost seat capacity, which remains the only consistent check against the pricing power of dominant hub carriers.